### `Maximal Marginal Relevance`

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

/home/tanishq/miniconda3/envs/interview/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
documents = [

    # Extremely similar cluster
    "Python tutorial for beginners",
    "Learn Python programming from scratch",
    "Complete Python coding course for beginners",
    "Python basics explained step by step",
    "Beginner friendly Python guide",

    # Another related cluster
    "Advanced Python for machine learning",
    "Python libraries for AI development",
    "Deep learning using Python and PyTorch",

    # Different but still somewhat relevant
    "Distributed systems engineering concepts",
    "Database indexing and query optimization",
    "System design interview preparation",

    # Completely different
    "Best cooking recipes for Italian pasta",
    "History of the Roman Empire",
    "Football tactics and formations analysis",
]

query = "python programming and coding"

In [15]:
model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = model.encode(documents)
query_embedding = model.encode([query])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2503.71it/s]


In [16]:
similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

top_k = 5

top_indices = np.argsort(similarities)[::-1][:top_k]

print("\n===== NORMAL TOP-K RETRIEVAL =====\n")

for idx in top_indices:
    print(f"{similarities[idx]:.4f}  |  {documents[idx]}")



===== NORMAL TOP-K RETRIEVAL =====

0.7970  |  Complete Python coding course for beginners
0.7290  |  Learn Python programming from scratch
0.6689  |  Beginner friendly Python guide
0.6603  |  Python tutorial for beginners
0.6421  |  Python basics explained step by step


In [12]:
x = [1, 3, 4, 5, 2] 
np.argsort(x) # gives indices that would sort the array


array([0, 4, 1, 2, 3])

In [23]:

def mmr(query_emb, doc_embs, docs, k=5, lambda_param=0.7):

    selected = []
    selected_indices = []

    candidate_indices = list(range(len(docs)))

    # First: most relevant document
    sim_to_query = cosine_similarity(query_emb, doc_embs)[0]

    first_idx = np.argmax(sim_to_query)

    selected.append(docs[first_idx])
    selected_indices.append(first_idx)

    candidate_indices.remove(first_idx)

    # Iteratively select
    while len(selected) < k:

        mmr_scores = []

        for idx in candidate_indices:

            relevance = sim_to_query[idx]

            selected_embs = doc_embs[selected_indices]

            similarity_to_selected = cosine_similarity(
                [doc_embs[idx]],
                selected_embs
            )[0]

            redundancy = max(similarity_to_selected)

            score = (
                lambda_param * relevance
                - (1 - lambda_param) * redundancy
            )

            mmr_scores.append((score, idx))
            # print(f"\nCandidate: {docs[idx]}")
            # print(f"Relevance: {relevance:.4f}")
            # print(f"Redundancy: {redundancy:.4f}")
            # print(f"MMR Score: {score:.4f}")

        best_score, best_idx = max(mmr_scores)

        selected.append(docs[best_idx])
        selected_indices.append(best_idx)

        candidate_indices.remove(best_idx)

    return selected_indices


mmr_indices = mmr(
    query_embedding,
    doc_embeddings,
    documents,
    k=5,
    lambda_param=0.5
)

print("\n===== MMR RETRIEVAL =====\n")

for idx in mmr_indices:
    print(f"{similarities[idx]:.4f}  |  {documents[idx]}")


===== MMR RETRIEVAL =====

0.7970  |  Complete Python coding course for beginners
0.6421  |  Python basics explained step by step
0.1883  |  Distributed systems engineering concepts
0.1025  |  Database indexing and query optimization
0.3650  |  Deep learning using Python and PyTorch
